# AirSense-IQ — Bengaluru Data Preprocessing (auto-discovery)

Drop raw CPCB station files into this notebook's folder (any mix of
`.csv` / `.xlsx`, any number of files per station, any year) and re-run
top to bottom. No code edits needed per file — station and format are
detected automatically.

**Output:** one combined dataset — `Date, Station, PM2.5, PM10, NO2, SO2, CO, Ozone`
— as both `.csv` and a single-sheet `.xlsx`.

**Target stations (6):** BTM Layout, Jayanagar 5th Block, Silk Board, Hebbal, Peenya, Kasturi Nagar
**Study period:** 01 Jul 2023 → 30 Jun 2026 (1,096 days/station, daily 24h average)
**Pollutants:** PM2.5, PM10, NO2, SO2, CO, O3 (Ozone)

Weather variables (RH, WS, WD, SR, BP, AT, RF) are intentionally **excluded**
from this pass — parked for later.

## Raw file formats this notebook understands

Two CPCB export formats have been seen so far, and both are auto-detected —
by file extension first, then by content:

1. **`cpcb_csv`** — a `Timestamp` column, one row per day, month-first dates
   (`1/1/2023`, sometimes with a trailing time like `1/1/2025 0:00`).
   No station name inside the file — it's inferred from the **filename**.
2. **`cpcb_xlsx`** — a CPCB portal export: 10-row metadata header
   (state/city/**station name**/parameter list), then a `From Date`/`To Date`
   table, day-first dates (`01-01-2026 00:00`). Station name is read directly
   **from the file's own metadata**, not the filename.

If a new file doesn't match either shape, the loader raises a clear error
naming the file and what it found, rather than silently mis-reading it.

In [9]:
import pandas as pd
import numpy as np
import os
import glob
import re
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [10]:
# ── Study window ───────────────────────────────────────────────
STUDY_START = pd.Timestamp("2023-07-01")
STUDY_END   = pd.Timestamp("2026-06-30")

# ── Required output pollutants ────────────────────────────────────
TARGETS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "Ozone"]

# ── Where to look for raw files, and where to write output ───────
RAW_DIR    = "../data/raw"                        # this notebook's folder — drop new files here (same directory as the .ipynb)
os.makedirs(RAW_DIR, exist_ok=True)
OUT_DIR    = "../data/processed"
OUT_CSV    = f"{OUT_DIR}/Bengaluru_AirQuality_combined.csv"
OUT_XLSX   = f"{OUT_DIR}/Bengaluru_AirQuality_combined.xlsx"
os.makedirs(OUT_DIR, exist_ok=True)

# Files to ignore when scanning RAW_DIR (our own outputs, checkpoints, etc.)
IGNORE_PATTERNS = ["Bengaluru_AirQuality_combined", ".ipynb_checkpoints"]

# ── Canonical station names + keywords used to recognise them ────
# Matching is done on a normalised filename (lowercased, spaces/underscores/
# hyphens stripped), so "silk_board.csv", "SilkBoard2024.csv" and
# "silk-board-24.csv" all resolve to the same station.
STATION_KEYWORDS = {
    "BTM Layout":          ["btm"],
    "Jayanagar 5th Block": ["jayanagar", "jaynagar"],  # "jaynagar" = common misspelling seen in raw filenames
    "Silk Board":          ["silkboard", "silk"],
    "Hebbal":              ["hebbal"],
    "Peenya":              ["peenya"],
    "Kasturi Nagar":       ["kasturi"],
}

# Safety valve: force a specific file to a specific station if the
# automatic filename match ever gets it wrong. Example:
#   MANUAL_STATION_OVERRIDE = {"ambiguous_file.csv": "Silk Board"}
MANUAL_STATION_OVERRIDE = {}

print(f"Study window : {STUDY_START.date()} -> {STUDY_END.date()}  "
      f"({(STUDY_END - STUDY_START).days + 1} days)")
print(f"Target pollutants : {TARGETS}")
print(f"Target stations    : {list(STATION_KEYWORDS.keys())}")

Study window : 2023-07-01 -> 2026-06-30  (1096 days)
Target pollutants : ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'Ozone']
Target stations    : ['BTM Layout', 'Jayanagar 5th Block', 'Silk Board', 'Hebbal', 'Peenya', 'Kasturi Nagar']


## 1. Discover raw files

Scans `RAW_DIR` for every `.csv` / `.xlsx` / `.xls`, skipping this notebook's
own output files. Each file's **format** is set by its extension, and its
**station** is inferred — from the filename for CSVs, from the file's own
metadata for xlsx portal exports.

In [11]:
def normalise(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def infer_format(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return "cpcb_csv"
    if ext in (".xlsx", ".xls"):
        return "cpcb_xlsx"
    raise ValueError(f"Unrecognised file extension for {path}")


def infer_station_from_filename(path):
    norm = normalise(os.path.basename(path))
    for station, keywords in STATION_KEYWORDS.items():
        if any(kw in norm for kw in keywords):
            return station
    return None


def infer_station_from_xlsx_metadata(path):
    """CPCB portal exports carry the station name in their own header
    block, e.g. row: 'Station', NaN, 'BTM Layout, Bengaluru - CPCB'.
    Matching on the label is case/whitespace-insensitive, and the value
    is taken as the first non-empty cell after the label (not assumed
    to sit in a fixed column), since exports vary slightly in layout."""
    raw = pd.read_excel(path, header=None, nrows=10)
    label_col = raw[0].astype(str).str.strip().str.lower()
    station_row = raw[label_col == "station"]
    if station_row.empty:
        return None

    row_vals = station_row.iloc[0]
    candidates = [v for v in row_vals.iloc[1:] if pd.notna(v) and str(v).strip() != ""]
    if not candidates:
        return None

    raw_name = str(candidates[0])
    norm = normalise(raw_name)
    for station, keywords in STATION_KEYWORDS.items():
        if any(kw in norm for kw in keywords):
            return station
    # Fall back to the raw metadata string (strip the ", Bengaluru - CPCB" suffix)
    return raw_name.split(",")[0].strip()


def discover_files():
    candidates = []
    for ext in ("*.csv", "*.xlsx", "*.xls"):
        candidates += glob.glob(os.path.join(RAW_DIR, ext))
    candidates = [c for c in candidates
                  if not any(p in os.path.basename(c) for p in IGNORE_PATTERNS)]

    rows = []
    for path in sorted(candidates):
        fname = os.path.basename(path)
        fmt = infer_format(path)

        if fname in MANUAL_STATION_OVERRIDE:
            station = MANUAL_STATION_OVERRIDE[fname]
            source = "manual override"
        elif fmt == "cpcb_xlsx":
            station = infer_station_from_xlsx_metadata(path)
            source = "xlsx metadata"
        else:
            station = infer_station_from_filename(path)
            source = "filename"

        rows.append({"file": fname, "path": path, "format": fmt,
                      "inferred_station": station, "inferred_from": source})

    return pd.DataFrame(rows, columns=["file", "path", "format", "inferred_station", "inferred_from"])


discovery = discover_files()

# ── Diagnostics: show exactly where we looked and what's there ───
abs_raw_dir = os.path.abspath(RAW_DIR)
print(f"Scanning RAW_DIR : {abs_raw_dir}")
all_files_here = os.listdir(RAW_DIR) if os.path.isdir(RAW_DIR) else []
print(f"All files/folders in that directory ({len(all_files_here)}): {all_files_here}")

if discovery.empty:
    print()
    print("[ERROR] No .csv / .xlsx / .xls files were found in RAW_DIR.")
    print("        Fix: upload your raw station files into the folder printed above")
    print("        (e.g. in Colab, use the Files pane on the left to upload directly")
    print("        into that folder, or run RAW_DIR = \"/content\" if you uploaded to the")
    print("        default Colab root), then re-run this cell.")

discovery


Scanning RAW_DIR : /content/aqi_data
All files/folders in that directory (24): ['peenya_2023.csv', 'silkboard_2025.csv', 'btm_2024.csv', 'hebbal_2023.csv', 'peenya_2024.csv', 'hebbal_2026.xlsx', 'peenya_2025.csv', 'silkboard_2024.csv', 'btm_2025.csv', 'peenya_2026.xlsx', 'kasturinagar_2024.csv', 'jaynagar_2023.csv', 'jaynagar_2026.xlsx', 'btm_2023.csv', 'hebbal_2025.csv', 'jaynagar_2025.csv', 'kasturinagar_2025.csv', 'silkboard_2026.xlsx', 'btm_2026.xlsx', 'hebbal_2024.csv', 'kasturinagar_2023.csv', 'jaynagar_2024.csv', 'kasturinagar_2026.xlsx', 'silkboard_2023.csv']


,file,path,format,inferred_station,inferred_from
0,btm_2023.csv,/content/aqi_data/btm_2023.csv,cpcb_csv,BTM Layout,filename
1,btm_2024.csv,/content/aqi_data/btm_2024.csv,cpcb_csv,BTM Layout,filename
2,btm_2025.csv,/content/aqi_data/btm_2025.csv,cpcb_csv,BTM Layout,filename
3,btm_2026.xlsx,/content/aqi_data/btm_2026.xlsx,cpcb_xlsx,BTM Layout,xlsx metadata
4,hebbal_2023.csv,/content/aqi_data/hebbal_2023.csv,cpcb_csv,Hebbal,filename
5,hebbal_2024.csv,/content/aqi_data/hebbal_2024.csv,cpcb_csv,Hebbal,filename
6,hebbal_2025.csv,/content/aqi_data/hebbal_2025.csv,cpcb_csv,Hebbal,filename
7,hebbal_2026.xlsx,/content/aqi_data/hebbal_2026.xlsx,cpcb_xlsx,Hebbal,xlsx metadata
8,jaynagar_2023.csv,/content/aqi_data/jaynagar_2023.csv,cpcb_csv,Jayanagar 5th Block,filename
9,jaynagar_2024.csv,/content/aqi_data/jaynagar_2024.csv,cpcb_csv,Jayanagar 5th Block,filename


In [12]:
if discovery.empty:
    print("[ERROR] `discovery` is empty — no CSV/xlsx files were found in RAW_DIR at all.")
    print("        Go back to the discovery cell above, check the printed RAW_DIR path,")
    print("        and confirm your raw files are actually sitting there.")
else:
    unmatched = discovery[discovery["inferred_station"].isna()]
    if len(unmatched):
        print("[WARN] Could not infer a station for these files — add them to "
              "MANUAL_STATION_OVERRIDE above and re-run:")
        print(unmatched[["file"]].to_string(index=False))
    else:
        print("All files matched to a station.")

    print()
    print("Files found per station:")
    print(discovery.groupby("inferred_station")["file"].apply(list).to_string())

All files matched to a station.

Files found per station:
inferred_station
BTM Layout             [btm_2023.csv, btm_2024.csv, btm_2025.csv, btm...
Hebbal                 [hebbal_2023.csv, hebbal_2024.csv, hebbal_2025...
Jayanagar 5th Block    [jaynagar_2023.csv, jaynagar_2024.csv, jaynaga...
Kasturi Nagar          [kasturinagar_2023.csv, kasturinagar_2024.csv,...
Peenya                 [peenya_2023.csv, peenya_2024.csv, peenya_2025...
Silk Board             [silkboard_2023.csv, silkboard_2024.csv, silkb...


## 2. Loaders

One loader per raw format. Column naming is standardised via `RENAME_MAP` so
the same `TARGETS` list works across formats.

In [13]:
RENAME_MAP = {
    "PM2.5 (µg/m³)": "PM2.5", "PM10 (µg/m³)": "PM10",
    "NO2 (µg/m³)": "NO2", "SO2 (µg/m³)": "SO2", "CO (mg/m³)": "CO",
    "Ozone (µg/m³)": "Ozone",
    # xlsx header variant
    "OZONE": "Ozone",
}

# Real-world exports vary in whitespace and in which unicode "micro" sign
# they use (µ = MICRO SIGN U+00B5 vs μ = GREEK SMALL LETTER MU U+03BC),
# so build the lookup from a normalised (stripped, mu-unified) column name
# rather than requiring an exact string match.
def _normalise_col(col):
    return str(col).strip().replace("μ", "µ")  # μ -> µ

_NORMALISED_RENAME_MAP = {_normalise_col(k): v for k, v in RENAME_MAP.items()}


def standardise_columns(df):
    """Strip whitespace / unify micro-sign variants, then rename to the
    canonical TARGETS names. Falls back to matching on the pollutant
    name alone (ignoring the unit in parentheses) if an exact header
    match isn't found, so small formatting differences don't silently
    drop a column."""
    df = df.rename(columns=lambda c: _normalise_col(c))
    df = df.rename(columns=_NORMALISED_RENAME_MAP)

    # Fallback: for any TARGET still missing, look for a column whose
    # name starts with the pollutant name (ignoring units/case/spacing).
    for target in TARGETS:
        if target in df.columns:
            continue
        target_key = target.lower().replace(".", "").replace(" ", "")
        for col in df.columns:
            col_key = re.sub(r"\(.*?\)", "", str(col)).lower().replace(".", "").replace(" ", "").strip()
            if col_key == target_key:
                df = df.rename(columns={col: target})
                break
    return df


def _drop_bad_dates(df, path):
    bad = df["Date"].isna().sum()
    if bad:
        print(f"  [NOTE] {os.path.basename(path)}: dropped {bad} row(s) with an unparseable/blank date")
        df = df[df["Date"].notna()]
    return df


def load_cpcb_csv(path):
    """Timestamp column, month-first dates, with or without a trailing
    time component."""
    df = pd.read_csv(path)
    df = standardise_columns(df)
    if "Timestamp" not in df.columns:
        raise ValueError(
            f"{path}: expected a 'Timestamp' column for cpcb_csv format, "
            f"found columns: {df.columns.tolist()}"
        )
    df["Date"] = pd.to_datetime(df["Timestamp"], format="mixed", dayfirst=False, errors="coerce")
    df["Date"] = df["Date"].dt.normalize()
    df = _drop_bad_dates(df, path)
    keep = ["Date"] + [c for c in TARGETS if c in df.columns]
    missing = [c for c in TARGETS if c not in df.columns]
    if missing:
        print(f"  [NOTE] {os.path.basename(path)}: missing columns {missing}")
    return df[keep]


def load_cpcb_xlsx(path):
    """CPCB portal export: metadata header block, then a From Date/To Date
    table, day-first dates."""
    raw = pd.read_excel(path, header=None)
    label_col = raw[0].astype(str).str.strip().str.lower()
    header_matches = raw[label_col == "from date"].index
    if len(header_matches) == 0:
        raise ValueError(
            f"{path}: could not find a 'From Date' header row. "
            f"First column's values (first 15 rows): {raw[0].head(15).tolist()}"
        )
    header_row = header_matches[0]
    df = pd.read_excel(path, header=header_row)

    # The header row's own casing/whitespace can vary ("from date",
    # "From Date "), so canonicalise it before renaming pollutant columns.
    df = df.rename(columns={c: "From Date" for c in df.columns
                             if str(c).strip().lower() == "from date"})
    df = standardise_columns(df)

    if "From Date" not in df.columns:
        raise ValueError(
            f"{path}: expected a 'From Date' column after the header row, "
            f"found columns: {df.columns.tolist()}"
        )

    # Day-first dates are the norm for this export, but formatting can
    # drift row-to-row (missing time component, different separators),
    # so try the strict expected format first and fall back to a more
    # lenient parse for whatever the strict pass couldn't handle.
    strict = pd.to_datetime(df["From Date"], format="%d-%m-%Y %H:%M", errors="coerce")
    still_missing = strict.isna() & df["From Date"].notna()
    if still_missing.any():
        lenient = pd.to_datetime(df.loc[still_missing, "From Date"],
                                  dayfirst=True, format="mixed", errors="coerce")
        strict.loc[still_missing] = lenient
    df["Date"] = strict.dt.normalize()
    df = _drop_bad_dates(df, path)

    keep = ["Date"] + [c for c in TARGETS if c in df.columns]
    missing = [c for c in TARGETS if c not in df.columns]
    if missing:
        print(f"  [NOTE] {os.path.basename(path)}: missing columns {missing}")
    return df[keep]


LOADERS = {"cpcb_csv": load_cpcb_csv, "cpcb_xlsx": load_cpcb_xlsx}


## 3. Clean each station

For every station with at least one discovered file:
1. Load and concatenate all its files
2. Drop duplicate dates, reindex to a **full daily calendar** so any missing
   day becomes an explicit gap rather than a silent one
3. **Interpolate** missing pollutant values (time-based, then edge-fill) —
   amount of imputation is reported per column, not hidden
4. Trim to the study window `2023-07-01` → `2026-06-30`

In [14]:
def clean_station(name, files_df):
    frames = []
    for _, row in files_df.iterrows():
        frames.append(LOADERS[row["format"]](row["path"]))
    d = pd.concat(frames, ignore_index=True)

    # Guarantee every target pollutant column exists, even if a station's
    # raw files never contained it at all (pd.concat only creates a column
    # if at least one frame had it — without this, d[TARGETS] below would
    # KeyError instead of just reporting it as fully missing).
    missing_entirely = [c for c in TARGETS if c not in d.columns]
    for c in missing_entirely:
        d[c] = np.nan
    if missing_entirely:
        print(f"  [NOTE] {name}: column(s) missing from every raw file: {missing_entirely}")

    d = d.sort_values("Date").drop_duplicates(subset="Date", keep="first")
    d = d.set_index("Date").asfreq("D")

    missing_before = d[TARGETS].isna().sum()
    was_missing = d[TARGETS].isna()          # remember which cells were real vs. absent, per pollutant, before we touch them

    d[TARGETS] = d[TARGETS].interpolate(method="time", limit_direction="both")
    d[TARGETS] = d[TARGETS].ffill().bfill()
    missing_after = d[TARGETS].isna().sum()

    # ── Imputation flags ────────────────────────────────────────────
    # For every pollutant, two extra columns travel alongside the value:
    #   <col>_is_imputed      True  = this value is estimated, not measured
    #   <col>_impute_run_len  length (days) of the consecutive gap this
    #                         value was filled from; 0 for real measurements.
    # A 2-day interpolated run and a 45-day ffill/bfill run both show up as
    # "imputed", but the run length tells them apart — filter on it
    # downstream (e.g. run_len <= 7) instead of guessing from is_imputed alone.
    for col in TARGETS:
        miss = was_missing[col]
        run_id = (miss != miss.shift()).cumsum()
        run_len = miss.groupby(run_id).transform("sum")
        d[f"{col}_is_imputed"] = miss.values
        d[f"{col}_impute_run_len"] = np.where(miss, run_len, 0).astype(int)

    full_min, full_max = d.index.min(), d.index.max()
    d = d.loc[(d.index >= STUDY_START) & (d.index <= STUDY_END)]
    d = d.reset_index().rename(columns={"index": "Date"})
    d.insert(1, "Station", name)

    report = pd.DataFrame({"missing_before_impute": missing_before,
                            "missing_after_impute": missing_after})
    # Longest imputed run per pollutant, restricted to the study window —
    # surfaces cases like a 45-day ffill/bfill stretch at a glance.
    run_cols = [f"{c}_impute_run_len" for c in TARGETS]
    report["max_impute_run_len"] = d[run_cols].max().values if len(d) else 0

    coverage_note = None
    if pd.isna(full_min) or pd.isna(full_max):
        coverage_note = "no valid dated rows were found for this station after cleaning"
    elif full_min > STUDY_START or full_max < STUDY_END:
        coverage_note = (f"raw data only spans {full_min.date()} -> {full_max.date()}, "
                          f"outside that range within the study window has no source file")
    return d, report, coverage_note


In [15]:
cleaned = {}
reports = {}
coverage_notes = {}

for station, files_df in discovery.dropna(subset=["inferred_station"]).groupby("inferred_station"):
    print(f"Processing {station}  ({len(files_df)} file(s): {files_df['file'].tolist()}) ...")
    d, rep, note = clean_station(station, files_df)
    cleaned[station] = d
    reports[station] = rep
    coverage_notes[station] = note
    if len(d):
        print(f"  rows: {len(d)}   date range: {d['Date'].min().date()} -> {d['Date'].max().date()}")
    else:
        print("  rows: 0   (no rows survived cleaning for this station — see [WARN] below)")
    if rep.values.sum() > 0:
        print(rep[rep["missing_before_impute"] > 0].to_string())
    if note:
        print(f"  [WARN] {note}")
    print()

missing_stations = [s for s in STATION_KEYWORDS if s not in cleaned]
if missing_stations:
    print(f"[NOTE] No files found yet for: {missing_stations}. "
          f"Drop their raw files into this folder and re-run.")


Processing BTM Layout  (4 file(s): ['btm_2023.csv', 'btm_2024.csv', 'btm_2025.csv', 'btm_2026.xlsx']) ...
  rows: 1096   date range: 2023-07-01 -> 2026-06-30
       missing_before_impute  missing_after_impute  max_impute_run_len
PM2.5                     86                     0                  57
PM10                      94                     0                  57
NO2                       77                     0                  24
SO2                       70                     0                  24
CO                        77                     0                  24
Ozone                     65                     0                  24

Processing Hebbal  (4 file(s): ['hebbal_2023.csv', 'hebbal_2024.csv', 'hebbal_2025.csv', 'hebbal_2026.xlsx']) ...
  rows: 1096   date range: 2023-07-01 -> 2026-06-30
       missing_before_impute  missing_after_impute  max_impute_run_len
PM2.5                     50                     0                  12
PM10                      49        

## 4. Combine all discovered stations into a single dataset

In [16]:
if not cleaned:
    raise RuntimeError(
        "`cleaned` is empty — no station had any files matched to it. "
        "Scroll up to the discovery/matching cells: check RAW_DIR contains your raw "
        "files, and that each filename/xlsx-metadata matched one of the 6 station "
        "keywords (or add a MANUAL_STATION_OVERRIDE entry and re-run from there)."
    )

FLAG_COLS = [f"{c}_is_imputed" for c in TARGETS] + [f"{c}_impute_run_len" for c in TARGETS]

bengaluru_df = pd.concat(cleaned.values(), ignore_index=True)
bengaluru_df = bengaluru_df[["Date", "Station"] + TARGETS + FLAG_COLS]
bengaluru_df = bengaluru_df.sort_values(["Station", "Date"]).reset_index(drop=True)

print(f"Stations combined : {sorted(cleaned.keys())}  ({len(cleaned)}/6)")
bengaluru_df.head()


Stations combined : ['BTM Layout', 'Hebbal', 'Jayanagar 5th Block', 'Kasturi Nagar', 'Peenya', 'Silk Board']  (6/6)


,Date,Station,PM2.5,PM10,NO2,SO2,CO,Ozone,PM2.5_is_imputed,PM10_is_imputed,NO2_is_imputed,SO2_is_imputed,CO_is_imputed,Ozone_is_imputed,PM2.5_impute_run_len,PM10_impute_run_len,NO2_impute_run_len,SO2_impute_run_len,CO_impute_run_len,Ozone_impute_run_len
0,2023-07-01,BTM Layout,13.841146,29.449722,24.417604,8.050000,0.736667,20.717396,False,False,False,False,False,False,0,0,0,0,0,0
1,2023-07-02,BTM Layout,13.007500,27.670978,24.414130,8.113261,0.732717,21.103977,False,False,False,False,False,False,0,0,0,0,0,0
2,2023-07-03,BTM Layout,8.396354,17.865000,24.442396,8.063437,0.739062,21.568229,False,False,False,False,False,False,0,0,0,0,0,0
3,2023-07-04,BTM Layout,6.002500,12.769583,24.432188,8.062396,0.739896,21.694688,False,False,False,False,False,False,0,0,0,0,0,0
4,2023-07-05,BTM Layout,6.888646,14.655938,24.435104,8.061667,0.740000,21.337500,False,False,False,False,False,False,0,0,0,0,0,0


## 5. Validation

Expected row count, duplicate (Station, Date) pairs, remaining nulls, and
per-station date coverage — checked before saving.

In [17]:
expected_days = (STUDY_END - STUDY_START).days + 1
print(f"Expected rows per fully-covered station : {expected_days}")
print()

print("Rows per station:")
print(bengaluru_df.groupby("Station").size())
print()

dupes = bengaluru_df.duplicated(subset=["Station", "Date"]).sum()
print(f"Duplicate (Station, Date) pairs : {dupes}")

nulls = bengaluru_df[TARGETS].isna().sum().sum()
print(f"Remaining null pollutant values : {nulls}")
print()

print("Date range per station:")
print(bengaluru_df.groupby("Station")["Date"].agg(["min", "max", "count"]))
print()

# ── Imputation summary ──────────────────────────────────────────────
# % of days imputed per station/pollutant, and the longest single
# ffill/bfill run — the things worth flagging before this goes into
# modelling (e.g. Kasturi Nagar's Ozone run).
print("Share of days imputed, per station x pollutant (%):")
imputed_share = (bengaluru_df.groupby("Station")[[f"{c}_is_imputed" for c in TARGETS]]
                  .mean().mul(100).round(1))
imputed_share.columns = TARGETS
print(imputed_share)
print()

print("Longest imputed run (days), per station x pollutant:")
max_run = (bengaluru_df.groupby("Station")[[f"{c}_impute_run_len" for c in TARGETS]]
           .max())
max_run.columns = TARGETS
print(max_run)

LONG_RUN_THRESHOLD = 14  # days — flag anything filled from a gap longer than this
long_runs = max_run[(max_run > LONG_RUN_THRESHOLD).any(axis=1)]
if len(long_runs):
    print(f"\n[WARN] Station x pollutant pairs with an imputed run longer than "
          f"{LONG_RUN_THRESHOLD} days (low-confidence fills — consider excluding "
          f"via the *_is_imputed / *_impute_run_len columns for model training/eval):")
    print(long_runs[(max_run > LONG_RUN_THRESHOLD).any(axis=1)])


Expected rows per fully-covered station : 1096

Rows per station:
Station
BTM Layout             1096
Hebbal                 1096
Jayanagar 5th Block    1096
Kasturi Nagar          1096
Peenya                 1096
Silk Board             1096
dtype: int64

Duplicate (Station, Date) pairs : 0
Remaining null pollutant values : 0

Date range per station:
                           min        max  count
Station                                         
BTM Layout          2023-07-01 2026-06-30   1096
Hebbal              2023-07-01 2026-06-30   1096
Jayanagar 5th Block 2023-07-01 2026-06-30   1096
Kasturi Nagar       2023-07-01 2026-06-30   1096
Peenya              2023-07-01 2026-06-30   1096
Silk Board          2023-07-01 2026-06-30   1096

Share of days imputed, per station x pollutant (%):
                     PM2.5  PM10   NO2   SO2    CO  Ozone
Station                                                  
BTM Layout             7.5   8.2   7.0   6.4   7.0    5.9
Hebbal                 2.9  

## 6. Save combined dataset (CSV + single-sheet Excel)

In [18]:
bengaluru_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    bengaluru_df.to_excel(writer, sheet_name="Bengaluru_AQ", index=False)

print(f"Saved : {OUT_CSV}")
print(f"Saved : {OUT_XLSX}  (sheet: 'Bengaluru_AQ')")
print(f"Shape : {bengaluru_df.shape}")
print(f"Stations included : {sorted(bengaluru_df['Station'].unique().tolist())}")

bengaluru_df.sample(min(5, len(bengaluru_df)), random_state=1).sort_values(["Station", "Date"])

Saved : data/Bengaluru_AirQuality_combined.csv
Saved : data/Bengaluru_AirQuality_combined.xlsx  (sheet: 'Bengaluru_AQ')
Shape : (6576, 20)
Stations included : ['BTM Layout', 'Hebbal', 'Jayanagar 5th Block', 'Kasturi Nagar', 'Peenya', 'Silk Board']


,Date,Station,PM2.5,PM10,NO2,SO2,CO,Ozone,PM2.5_is_imputed,PM10_is_imputed,NO2_is_imputed,SO2_is_imputed,CO_is_imputed,Ozone_is_imputed,PM2.5_impute_run_len,PM10_impute_run_len,NO2_impute_run_len,SO2_impute_run_len,CO_impute_run_len,Ozone_impute_run_len
2836,2025-04-05,Jayanagar 5th Block,19.120000,42.780000,13.160000,4.430000,0.160000,5.980000,False,False,False,False,False,False,0,0,0,0,0,0
3334,2023-08-16,Kasturi Nagar,36.054063,74.364063,9.005208,5.420521,0.361771,27.487292,False,False,False,False,False,False,0,0,0,0,0,0
4179,2025-12-08,Kasturi Nagar,36.870000,72.130000,23.510000,8.500000,1.090000,20.280000,False,False,False,False,False,False,0,0,0,0,0,0
5977,2024-11-09,Silk Board,14.970000,75.150000,18.120000,6.780000,0.660000,22.270000,False,False,False,False,False,False,0,0,0,0,0,0
6257,2025-08-16,Silk Board,12.410000,92.750000,7.700000,8.460000,0.670000,19.350000,False,False,False,False,False,False,0,0,0,0,0,0


## Next steps

- **Just drop more raw files into this notebook's folder and re-run
  top to bottom** — no code changes needed. New files are picked up
  automatically by extension + filename/metadata matching.
- If a file's station can't be auto-detected, it's listed under
  `unmatched` in step 1 — add an entry to `MANUAL_STATION_OVERRIDE` and re-run.
- Weather variables (RH, WS, WD, SR, BP, AT, RF) were intentionally left out
  of this pass; when ready they can be merged back in the same way (note:
  AT, RF, TOT-RF had **0% coverage** in BTM Layout's raw files, so those two
  will need an actual data source, not just imputation).
- Once all 6 stations are present, `Bengaluru_AirQuality_combined.csv/.xlsx`
  is the input to the feature-engineering step (lags, rolling stats,
  city-wide averages, AQI computation) that feeds `train_models.py`.